In [1]:
## 2026 CANADIAN GRAND PRIX — PODIUM PREDICTION
### Rolling data: Australia (R1) + China (R2) + Japan (R3) + Miami (R4)
### Canadian GP qualifying grid sourced live (May 2026)
### Weather: 60-90% rain probability, ~12-16°C race day

# ── Imports ───────────────────────────────────────────────────────────────────
import sys
!{sys.executable} -m pip install fastf1 --quiet

import fastf1
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneOut
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor

# ── Load Canadian GP qualifying session via FastF1 ────────────────────────────
session = fastf1.get_session(2026, "Canada", "Q")
session.load(telemetry=False, weather=True, messages=False)
laps = session.laps

def td_to_s(td):
    """Convert timedelta (or NaT) to float seconds."""
    if pd.isnull(td):
        return np.nan
    return td.total_seconds()

records = []
for driver in laps["Driver"].unique():
    drv_all = laps.pick_drivers(driver).pick_wo_box()
    if drv_all.empty:
        continue
    best_s1  = drv_all["Sector1Time"].dropna().min()
    best_s2  = drv_all["Sector2Time"].dropna().min()
    best_s3  = drv_all["Sector3Time"].dropna().min()
    best_lap = drv_all["LapTime"].dropna().min()
    records.append({
        "Driver":    driver,
        "S1":        td_to_s(best_s1),
        "S2":        td_to_s(best_s2),
        "S3":        td_to_s(best_s3),
        "BestLap_s": td_to_s(best_lap),
    })

quali_df = pd.DataFrame(records)
quali_df["UltimateLap_s"] = (quali_df["S1"] + quali_df["S2"] + quali_df["S3"]).round(3)

pole_time = quali_df["BestLap_s"].min()
quali_df["CanadaGapFromPole_s"]  = (quali_df["BestLap_s"] - pole_time).round(3)
quali_df["CanadaGapFromPolePct"] = (quali_df["CanadaGapFromPole_s"] / pole_time * 100).round(4)

quali_df = quali_df.sort_values("BestLap_s").reset_index(drop=True)
quali_df["CanadaGrid"] = quali_df.index + 1

# ── Race-day weather (sourced: RacingNews365 / F1.com / GPFans, May 24 2026) ──
# Race start 16:00 local; heavy rain morning, improving but 60-90% rain at race
rain_probability = 0.75   # consensus: 60-90% across forecasters
temperature      = 14.0   # ~12-16°C range on race day

quali_df["RainProbability"] = rain_probability
quali_df["Temperature"]     = temperature

# ── Driver standings after Miami GP (Round 4) ─────────────────────────────────
# Sources: SI.com, RacingNews365, formula1archive.com
driver_points = {
    "ANT": 100, "RUS": 80,  "LEC": 63,  "NOR": 51,
    "HAM": 49,  "PIA": 36,  "VER": 22,  "BEA": 17,
    "GAS": 15,  "LAW": 10,  "COL": 13,  "LIN": 8,
    "HAD": 4,   "SAI": 6,   "ALB": 5,   "BOR": 2,
    "OCO": 1,   "ALO": 0,   "STR": 0,   "PER": 0,
    "BOT": 0,   "HUL": 0,
}
# NOTE: Colapinto promoted to P7 post-penalty (+2 pts vs pre-penalty standings)
# Hamilton promoted to P6 post-penalty (+2 pts)

max_driver_pts = max(v for v in driver_points.values() if v > 0)
quali_df["DriverPoints"]     = quali_df["Driver"].map(driver_points).fillna(0)
quali_df["DriverPointsNorm"] = (quali_df["DriverPoints"] / max_driver_pts).round(4)

print(quali_df[["Driver", "S1", "S2", "S3", "UltimateLap_s",
                "BestLap_s", "CanadaGrid", "CanadaGapFromPole_s",
                "CanadaGapFromPolePct", "RainProbability", "Temperature",
                "DriverPointsNorm"]])
quali_df.to_csv("canada_quali.csv", index=False)
print("\nSaved → canada_quali.csv")


# ── Rolling features: AUS + CHN + JPN + MIA (4 races) ────────────────────────

# Finish positions
aus_finish = {
    "RUS": 1,  "ANT": 2,  "LEC": 3,  "HAM": 4,
    "NOR": 5,  "VER": 6,  "BEA": 7,  "LIN": 8,
    "BOR": 9,  "GAS": 10, "OCO": 11, "ALB": 12,
    "LAW": 13, "COL": 14, "SAI": 15, "PER": 16,
    "STR": 17, "ALO": 18, "BOT": 19, "HAD": 20,
    "PIA": 21, "HUL": 22,
}

china_finish = {
    "ANT": 1,  "RUS": 2,  "HAM": 3,  "LEC": 4,
    "BEA": 5,  "GAS": 6,  "LAW": 7,  "HAD": 8,
    "SAI": 9,  "COL": 10, "HUL": 11, "LIN": 12,
    "BOT": 13, "OCO": 14, "PER": 15, "VER": 16,
    "ALO": 17, "STR": 18, "NOR": 19, "BOR": 20,
    "ALB": 21, "PIA": 22,
}

japan_finish = {
    "ANT": 1,  "PIA": 2,  "LEC": 3,  "RUS": 4,
    "NOR": 5,  "HAM": 6,  "GAS": 7,  "VER": 8,
    "LAW": 9,  "OCO": 10, "HUL": 11, "HAD": 12,
    "BOR": 13, "LIN": 14, "SAI": 15, "COL": 16,
    "PER": 17, "ALO": 18, "BOT": 19, "ALB": 20,
    "STR": 21, "BEA": 22,   # STR & BEA DNF → last
}

# Miami finish (FINAL post-penalty classification)
# Sources: the-race.com, formula1archive.com, formula1.com
# HUL/LAW/GAS/HAD = DNF → ranked 19-22
miami_finish = {
    "ANT": 1,  "NOR": 2,  "PIA": 3,  "RUS": 4,
    "VER": 5,  "HAM": 6,  "COL": 7,  "LEC": 8,
    "SAI": 9,  "ALB": 10, "BEA": 11, "BOR": 12,
    "OCO": 13, "LIN": 14, "ALO": 15, "PER": 16,
    "STR": 17, "BOT": 18, "HUL": 19, "LAW": 20,
    "GAS": 21, "HAD": 22,
}

# Grid/qualifying positions
aus_grid = {
    "RUS": 1,  "ANT": 2,  "HAD": 3,  "LEC": 4,  "PIA": 5,
    "NOR": 6,  "HAM": 7,  "LAW": 8,  "LIN": 9,  "BOR": 10,
    "OCO": 11, "HUL": 12, "ALB": 13, "GAS": 14, "COL": 15,
    "BEA": 16, "ALO": 17, "PER": 18, "BOT": 19, "VER": 20,
    "SAI": 21, "STR": 22,
}

china_grid = {
    "ANT": 1,  "RUS": 2,  "HAM": 3,  "LEC": 4,  "PIA": 5,
    "NOR": 6,  "GAS": 7,  "VER": 8,  "HAD": 9,  "BEA": 10,
    "HUL": 11, "COL": 12, "OCO": 13, "LAW": 14, "LIN": 15,
    "BOR": 16, "SAI": 17, "ALB": 18, "ALO": 19, "BOT": 20,
    "STR": 21, "PER": 22,
}

japan_grid = {
    "ANT": 1,  "RUS": 2,  "PIA": 3,  "LEC": 4,  "NOR": 5,
    "HAM": 6,  "GAS": 7,  "HAD": 8,  "BOR": 9,  "LIN": 10,
    "VER": 11, "OCO": 12, "HUL": 13, "LAW": 14, "COL": 15,
    "SAI": 16, "ALB": 17, "BEA": 18, "PER": 19, "BOT": 20,
    "ALO": 21, "STR": 22,
}

# Miami qualifying (Hadjar DSQ → treated as P22 start effectively P9 on road)
# Using ADJUSTED grid (post-DSQ Hadjar — Bortoleto DSQ too in sprint but quali ok)
# Source: formula1.com race result / wikipedia
miami_grid = {
    "ANT": 1,  "NOR": 2,  "VER": 3,  "LEC": 4,  "PIA": 5,
    "RUS": 6,  "GAS": 7,  "BEA": 8,  "SAI": 9,  "OCO": 10,
    "HUL": 11, "LAW": 12, "LIN": 13, "ALB": 14, "BOR": 15,
    "COL": 16, "ALO": 17, "STR": 18, "BOT": 19, "HAM": 20,
    "HAD": 21, "PER": 22,
}

# Canadian GP qualifying grid (confirmed, May 23 2026)
# Source: the-race.com, crash.net, gpfans.com, racingnews365.com
# STR qualified P21 but starts from pit lane (ESME replacement)
canada_grid = {
    "RUS": 1,  "ANT": 2,  "NOR": 3,  "PIA": 4,  "HAM": 5,
    "VER": 6,  "HAD": 7,  "LEC": 8,  "LIN": 9,  "COL": 10,
    "HUL": 11, "LAW": 12, "BOR": 13, "GAS": 14, "SAI": 15,
    "BEA": 16, "OCO": 17, "ALB": 18, "ALO": 19, "PER": 20,
    "BOT": 21, "STR": 22,   # STR starts from pit lane
}

# Constructor standings after Miami (Round 4)
# Source: SI.com, RacingNews365
team_points = {
    "Mercedes":     180,
    "Ferrari":      112,
    "McLaren":       94,
    "Alpine":        23,
    "Haas":          18,
    "Red Bull":      26,
    "Racing Bulls":  18,
    "Audi":           2,
    "Williams":       8,
    "Cadillac":       0,
    "Aston Martin":   0,
}

driver_to_team = {
    "RUS": "Mercedes",     "ANT": "Mercedes",
    "HAM": "Ferrari",      "LEC": "Ferrari",
    "NOR": "McLaren",      "PIA": "McLaren",
    "VER": "Red Bull",     "HAD": "Red Bull",
    "BEA": "Haas",         "OCO": "Haas",
    "LAW": "Racing Bulls", "LIN": "Racing Bulls",
    "HUL": "Audi",         "BOR": "Audi",
    "GAS": "Alpine",       "COL": "Alpine",
    "SAI": "Williams",     "ALB": "Williams",
    "PER": "Cadillac",     "BOT": "Cadillac",
    "ALO": "Aston Martin", "STR": "Aston Martin",
}

# ── Build 4-race rolling features ─────────────────────────────────────────────
drivers = list(driver_to_team.keys())
N = len(drivers)  # 22

records = []
for drv in drivers:
    aus_fp  = aus_finish[drv];   chn_fp = china_finish[drv]
    jpn_fp  = japan_finish[drv]; mia_fp = miami_finish[drv]
    aus_gp  = aus_grid[drv];     chn_gp = china_grid[drv]
    jpn_gp  = japan_grid[drv];   mia_gp = miami_grid[drv]

    avg_finish_pos  = round((aus_fp + chn_fp + jpn_fp + mia_fp) / 4, 2)
    avg_finish_norm = round(
        ((aus_fp-1)/(N-1) + (chn_fp-1)/(N-1) +
         (jpn_fp-1)/(N-1) + (mia_fp-1)/(N-1)) / 4, 4
    )

    avg_grid_pos  = round((aus_gp + chn_gp + jpn_gp + mia_gp) / 4, 2)
    avg_grid_norm = round(
        ((aus_gp-1)/(N-1) + (chn_gp-1)/(N-1) +
         (jpn_gp-1)/(N-1) + (mia_gp-1)/(N-1)) / 4, 4
    )

    # DNF flags — positions toward the back typically signal DNF/DNS
    dnf_aus = 1 if aus_fp >= 18 else 0
    dnf_chn = 1 if chn_fp >= 16 else 0   # VER/ALO/STR DNF; NOR+ DNS
    dnf_jpn = 1 if jpn_fp >= 21 else 0   # STR/BEA DNF
    dnf_mia = 1 if mia_fp >= 19 else 0   # HUL/LAW/GAS/HAD DNF
    dnf_rate = round((dnf_aus + dnf_chn + dnf_jpn + dnf_mia) / 4, 3)

    # Recent form: weight later races more heavily (exponential decay 0.4/0.3/0.2/0.1)
    # Higher weight = more recent
    recent_finish_norm = round(
        0.4*(mia_fp-1)/(N-1) + 0.3*(jpn_fp-1)/(N-1) +
        0.2*(chn_fp-1)/(N-1) + 0.1*(aus_fp-1)/(N-1), 4
    )

    records.append({
        "Driver":            drv,
        "Team":              driver_to_team[drv],
        "AusFinish":         aus_fp, "ChinaFinish":  chn_fp,
        "JapanFinish":       jpn_fp, "MiamiFinish":  mia_fp,
        "AvgFinishPos":      avg_finish_pos,
        "AvgFinishNorm":     avg_finish_norm,
        "RecentFinishNorm":  recent_finish_norm,
        "AusGrid":           aus_gp, "ChinaGrid":    chn_gp,
        "JapanGrid_prev":    jpn_gp, "MiamiGrid_prev": mia_gp,
        "AvgGridPos":        avg_grid_pos,
        "AvgGridNorm":       avg_grid_norm,
        "DNF_rate":          dnf_rate,
    })

rolling_df = pd.DataFrame(records)

max_team_pts   = max(team_points.values())
max_driver_pts = max(driver_points.values())

rolling_df["TeamScore"] = rolling_df["Team"].map(
    {t: max(p, 0.5) / max_team_pts for t, p in team_points.items()}
).round(4)

rolling_df["DriverPointsNorm"] = rolling_df["Driver"].map(
    {d: p / max_driver_pts for d, p in driver_points.items()}
).round(4)

# Print rolling features summary
print("\n" + "=" * 105)
print("  ROLLING FEATURES FOR CANADIAN GP PREDICTION")
print("  Based on: Australia (R1) + China (R2) + Japan (R3) + Miami (R4)")
print("=" * 105)
print(f"\n{'DRV':<6}{'Team':<16}{'AUS':<5}{'CHN':<5}{'JPN':<5}{'MIA':<5}{'AvgFin':<9}"
      f"{'RecentNorm':<12}{'AvgGrid':<9}{'AvgGridNorm':<13}"
      f"{'DNF%':<7}{'TeamScore':<11}{'DrvPtsNorm'}")
print("─" * 110)
for _, r in rolling_df.sort_values("AvgFinishPos").iterrows():
    print(f"{r['Driver']:<6}{r['Team']:<16}{r['AusFinish']:<5}{r['ChinaFinish']:<5}"
          f"{r['JapanFinish']:<5}{r['MiamiFinish']:<5}{r['AvgFinishPos']:<9}"
          f"{r['RecentFinishNorm']:<12}{r['AvgGridPos']:<9}{r['AvgGridNorm']:<13}"
          f"{r['DNF_rate']:<7}{r['TeamScore']:<11}{r['DriverPointsNorm']}")

rolling_df.to_csv("rolling_features_aus_chn_jpn_mia.csv", index=False)
print("\nSaved → rolling_features_aus_chn_jpn_mia.csv")


# ── Load Sprint race pace (Canada is a Sprint weekend — no FP2) ───────────────
# Sprint provides best available race-pace signal for the GP
session_spr = fastf1.get_session(2026, "Canada", "Sprint")
session_spr.load(telemetry=False, weather=False, messages=False)

spr_laps = session_spr.laps.copy()
spr_laps["LapTime_s"] = spr_laps["LapTime"].apply(td_to_s)
spr_laps = spr_laps[spr_laps["IsAccurate"] == True]

def within_107(group):
    med = group["LapTime_s"].median()
    return group[group["LapTime_s"] <= med * 1.07]

spr_laps = spr_laps.groupby("Driver", group_keys=False).apply(within_107)

fp2_records = []
for driver, grp in spr_laps.groupby("Driver"):
    fp2_records.append({
        "Driver":           driver,
        "FP2_MedianPace_s": round(grp["LapTime_s"].median(), 3),
        "FP2_MeanPace_s":   round(grp["LapTime_s"].mean(),   3),
        "FP2_BestPace_s":   round(grp["LapTime_s"].min(),    3),
        "FP2_LapCount":     len(grp),
        "FP2_Compounds":    str(grp["Compound"].value_counts().to_dict()),
    })

fp2_df = pd.DataFrame(fp2_records).sort_values("FP2_MedianPace_s").reset_index(drop=True)
ref_pace = fp2_df["FP2_MedianPace_s"].min()
fp2_df["FP2_GapToFastest_s"] = (fp2_df["FP2_MedianPace_s"] - ref_pace).round(3)

print("\nSprint race pace (proxy for GP long-run pace):")
print(fp2_df[["Driver", "FP2_MedianPace_s", "FP2_GapToFastest_s",
              "FP2_LapCount", "FP2_Compounds"]])

fp2_df.to_csv("canada_sprint_pace.csv", index=False)
print("\nSaved → canada_sprint_pace.csv")


# ── Merge all data ────────────────────────────────────────────────────────────
df = quali_df.merge(
    rolling_df[["Driver", "AvgFinishNorm", "RecentFinishNorm", "AvgGridNorm",
                "DNF_rate", "TeamScore", "Team"]],
    on="Driver", how="left"
)

df_final = df.merge(
    fp2_df[["Driver", "FP2_GapToFastest_s"]],
    on="Driver", how="left"
)


# ── Feature Engineering ───────────────────────────────────────────────────────
feature_cols = [
    "UltimateLap_s",        # Canada quali: best S1+S2+S3
    "CanadaGapFromPole_s",  # Canada quali: gap to pole
    "CanadaGrid",           # Canada qualifying position
    "FP2_GapToFastest_s",   # Sprint long-run race pace (proxy for FP2)
    "TeamScore",            # Constructor points normalised (after Miami R4)
    "DriverPointsNorm",     # Driver points normalised (after Miami R4)
    "AvgFinishNorm",        # Rolling avg finish (AUS + CHN + JPN + MIA)
    "RecentFinishNorm",     # Weighted recent form (MIA 40%, JPN 30%, CHN 20%, AUS 10%)
    "AvgGridNorm",          # Rolling avg grid (4 races)
    "DNF_rate",             # Reliability signal (4 races)
    "RainProbability",      # Weather: 60-90% rain (using 0.75 consensus)
    "Temperature",          # Weather: ~14°C race day
]

# ── Target variable ───────────────────────────────────────────────────────────
# Weight recent form more heavily than plain average (3 races vs 4 races)
df_final["RaceScore"] = (
    0.5 * df_final["AvgFinishNorm"] +
    0.3 * df_final["RecentFinishNorm"] +
    0.2 * df_final["AvgGridNorm"]
).round(4)

X = df_final[feature_cols].copy()
y = df_final["RaceScore"]

imputer   = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)


# ── Multi-model LOO-CV comparison ─────────────────────────────────────────────
models = {
    "Ridge":         Ridge(alpha=1.0),
    "ElasticNet":    ElasticNet(alpha=0.1, l1_ratio=0.5),
    "RandomForest":  RandomForestRegressor(
                         n_estimators=100, max_depth=3,
                         min_samples_leaf=3, random_state=42),
    "GradientBoost": GradientBoostingRegressor(
                         n_estimators=100, learning_rate=0.05,
                         max_depth=2, subsample=0.8, random_state=42),
    "XGBoost":       XGBRegressor(
                         n_estimators=100, learning_rate=0.05,
                         max_depth=2, subsample=0.8,
                         colsample_bytree=0.8, reg_lambda=2.0,
                         random_state=42),
}

loo     = LeaveOneOut()
results = {}

for name, mdl in models.items():
    pipe = (Pipeline([("scaler", StandardScaler()), ("model", mdl)])
            if name in ["Ridge", "ElasticNet"]
            else Pipeline([("model", mdl)]))
    errors = []
    for train_idx, test_idx in loo.split(X_imputed):
        pipe.fit(X_imputed[train_idx], y.iloc[train_idx])
        errors.append(abs(pipe.predict(X_imputed[test_idx])[0] - y.iloc[test_idx].iloc[0]))
    results[name] = {"MAE": round(np.mean(errors), 4), "StdDev": round(np.std(errors), 4)}

print(f"\n{'Model':<18} {'LOO MAE':>8} {'Std Dev':>8}  {'Verdict'}")
print("─" * 55)
for name, res in sorted(results.items(), key=lambda x: x[1]["MAE"]):
    best = " ← best" if res["MAE"] == min(r["MAE"] for r in results.values()) else ""
    print(f"{name:<18} {res['MAE']:>8.4f} {res['StdDev']:>8.4f}  {best}")


# ── Best model final prediction ───────────────────────────────────────────────
best_name  = min(results, key=lambda x: results[x]["MAE"])
best_model = models[best_name]
final_pipe = (Pipeline([("scaler", StandardScaler()), ("model", best_model)])
              if best_name in ["Ridge", "ElasticNet"]
              else Pipeline([("model", best_model)]))
final_pipe.fit(X_imputed, y)
df_final["PredictedScore"] = final_pipe.predict(X_imputed)

final = df_final.sort_values("PredictedScore").reset_index(drop=True)

print("\n" + "=" * 72)
print("   PREDICTED FINISHING ORDER — 2026 CANADIAN GRAND PRIX")
print("   Rolling target: AUS + CHN + JPN + MIA avg finish + recent form")
print("=" * 72)
print(f"  {'P':<4}{'DRV':<6}{'Team':<22}{'Score'}")
print("  " + "─" * 48)
for i, row in final.iterrows():
    print(f"  {i+1:<4}{row['Driver']:<6}{row['Team']:<22}{row['PredictedScore']:.4f}")

podium = final.head(3)
print(f"\n  {'='*52}")
print(f"  🥇 P1: {podium.iloc[0]['Driver']} ({podium.iloc[0]['Team']})")
print(f"  🥈 P2: {podium.iloc[1]['Driver']} ({podium.iloc[1]['Team']})")
print(f"  🥉 P3: {podium.iloc[2]['Driver']} ({podium.iloc[2]['Team']})")
print(f"\n  Best model:   {best_name}")
print(f"  LOO MAE:      {results[best_name]['MAE']:.4f}")
print(f"  LOO Std Dev:  {results[best_name]['StdDev']:.4f}")
print(f"  Weather:      {df_final['Temperature'].iloc[0]:.1f}°C | "
      f"Rain: {df_final['RainProbability'].iloc[0]:.0%}")
print(f"  {'='*52}")


# ── Feature importances ───────────────────────────────────────────────────────
if best_name in ["RandomForest", "GradientBoost", "XGBoost"]:
    importances = final_pipe.named_steps["model"].feature_importances_
    print("\nFeature importances:")
    for feat, imp in sorted(zip(feature_cols, importances), key=lambda x: -x[1]):
        bar = "█" * int(imp * 50)
        print(f"  {feat:<30} {imp:.4f}  {bar}")
elif best_name in ["Ridge", "ElasticNet"]:
    coefficients = final_pipe.named_steps["model"].coef_
    print("\nFeature coefficients:")
    for feat, coef in sorted(zip(feature_cols, coefficients), key=lambda x: -abs(x[1])):
        bar = "█" * int(abs(coef) * 20)
        print(f"  {feat:<30} {coef:+.4f}  {bar}")


# ── All-models side-by-side comparison ───────────────────────────────────────
all_predictions = {}
for name, mdl in models.items():
    pipe = (Pipeline([("scaler", StandardScaler()), ("model", mdl)])
            if name in ["Ridge", "ElasticNet"]
            else Pipeline([("model", mdl)]))
    pipe.fit(X_imputed, y)
    ranked = pd.Series(pipe.predict(X_imputed),
                       index=df_final["Driver"]).rank(method="min").astype(int)
    all_predictions[name] = ranked

pred_df = pd.DataFrame(all_predictions)
pred_df["BestModelPos"] = pd.Series(
    final_pipe.predict(X_imputed), index=df_final["Driver"]
).rank(method="min").astype(int)
pred_df = pred_df.sort_values("BestModelPos")

print("\n" + "=" * 98)
print("   PREDICTED FINISHING ORDER — 2026 CANADIAN GRAND PRIX  (all models)")
print("=" * 98)
col_w  = 14
header = f"  {'DRV':<6}{'Team':<22}"
for name in models: header += f"{name:>{col_w}}"
header += f"  {'★ Best':>{col_w}}"
print(header)
print("  " + "─" * 90)
for driver in pred_df.index:
    team = df_final.loc[df_final["Driver"] == driver, "Team"].values[0]
    row  = f"  {driver:<6}{team:<22}"
    for name in models:
        row += f"  P{pred_df.loc[driver, name]:<{col_w-2}}"
    row += f"  P{pred_df.loc[driver, 'BestModelPos']:<{col_w-2}}"
    print(row)

# ── Consensus podium ──────────────────────────────────────────────────────────
best_order = pred_df.sort_values("BestModelPos")
p1, p2, p3 = best_order.index[0], best_order.index[1], best_order.index[2]
t1 = df_final.loc[df_final["Driver"] == p1, "Team"].values[0]
t2 = df_final.loc[df_final["Driver"] == p2, "Team"].values[0]
t3 = df_final.loc[df_final["Driver"] == p3, "Team"].values[0]

print(f"\n  {'='*60}")
print(f"  Best model: {best_name} (LOO MAE: {results[best_name]['MAE']:.4f})")
print(f"  🥇 P1: {p1} ({t1})")
print(f"  🥈 P2: {p2} ({t2})")
print(f"  🥉 P3: {p3} ({t3})")
print(f"  Weather:    {df_final['Temperature'].iloc[0]:.1f}°C | "
      f"Rain: {df_final['RainProbability'].iloc[0]:.0%}")
print(f"  {'='*60}")

print("\n  CONSENSUS PODIUM (across all models):")
print("  " + "─" * 40)
medals = ["🥇", "🥈", "🥉"]
for pos in [1, 2, 3]:
    pos_counts = {}
    for name in models:
        for d in pred_df[pred_df[name] == pos].index:
            pos_counts[d] = pos_counts.get(d, 0) + 1
    if pos_counts:
        top   = max(pos_counts, key=pos_counts.get)
        votes = pos_counts[top]
        print(f"  {medals[pos-1]} P{pos}: {top} — {votes}/{len(models)} models agree")
print(f"  {'='*60}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 1.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 14.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00


req         WARNING 	DEFAULT CACHE ENABLED! (24.0 KB) /root/.cache/fastf1
core           INFO 	Loading data for Canadian Grand Prix - Qualifying [v3.8.3]
INFO:fastf1.fastf1.core:Loading data for Canadian Grand Prix - Qualifying [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_statu

   Driver      S1      S2      S3  UltimateLap_s  BestLap_s  CanadaGrid  \
0     RUS  20.600  22.902  29.076         72.578     72.578           1   
1     ANT  20.547  23.061  28.919         72.527     72.646           2   
2     NOR  20.619  22.927  29.103         72.649     72.729           3   
3     PIA  20.774  22.926  29.081         72.781     72.781           4   
4     HAM  20.606  23.029  29.233         72.868     72.868           5   
5     VER  20.580  23.155  29.172         72.907     72.907           6   
6     HAD  20.549  23.218  29.148         72.915     72.935           7   
7     LEC  20.656  23.143  29.151         72.950     72.976           8   
8     LIN  20.654  23.309  29.278         73.241     73.280           9   
9     COL  20.997  23.407  29.250         73.654     73.697          10   
10    HUL  21.053  23.335  29.421         73.809     73.886          11   
11    LAW  20.886  23.528  29.445         73.859     73.897          12   
12    BOR  21.168  23.432

req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
INFO:fastf1.api:Fetching session status data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
INFO:fastf1.fastf1.req:No cached data found for lap_count. Loading data...
_api           INFO 	F


Sprint race pace (proxy for GP long-run pace):
   Driver  FP2_MedianPace_s  FP2_GapToFastest_s  FP2_LapCount  \
0     RUS            74.891               0.000            22   
1     ANT            74.904               0.013            22   
2     NOR            74.930               0.039            22   
3     PIA            75.154               0.263            22   
4     HAD            75.219               0.328            13   
5     HAM            75.233               0.342            22   
6     LEC            75.278               0.387            22   
7     VER            75.310               0.419            22   
8     LIN            75.998               1.107            22   
9     COL            76.067               1.176            22   
10    SAI            76.434               1.543            22   
11    GAS            76.904               2.013            18   
12    BEA            77.110               2.219            19   
13    LAW            77.148               